# AMPS I · Week 08 — Lab
## Newton's method by hand

| | |
|---|---|
| **Lecture sections** | Part A (residual, Gâteaux derivative, Newton, Picard) · Part B (Newton in DOLFINx) · Part D (LLM) |
| **Time** | ≈ 60 min (after the midterm; finish at home — HW8 in Week 9 builds on it) |
| **You will** | compute a Gâteaux derivative by hand and check it against `ufl.derivative` and a Taylor test · write a Newton loop with assembly, lifting and `ghostUpdate` · see quadratic vs linear convergence · check mesh independence and MMS rates of the converged solution |

Model problem: $-\nabla\cdot\bigl((1+u^2)\nabla u\bigr)=f$ on the unit square, $u=g$ on $\partial\Omega$, with the manufactured solution
$\tilde u=1+\sin(\pi x)\,y^2$.

**Reminder.** `%%fenicsx` cells run in a separate MPI process; ordinary cells have NumPy and matplotlib only. Data passes through `/tmp/fnx/`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · The Gâteaux derivative  *(lecture §A2)*
$F(u;v)=\int q(u)\nabla u\cdot\nabla v-\int fv$ with $q(u)=1+u^2$. By hand:
$J(u;w,v)=\int q(u)\nabla w\cdot\nabla v+\int q'(u)\,w\,\nabla u\cdot\nabla v$, $q'(u)=2u$.
Compare the assembled matrices with `ufl.derivative`, then run a **Taylor test**: for a direction $w$,
$\|F(u+\varepsilon w)-F(u)-\varepsilon J(u)w\|$ must be $O(\varepsilon^2)$ (ratio 4 when $\varepsilon$ halves); without the Jacobian term it is only $O(\varepsilon)$.

In [ ]:
%%fenicsx -np 1
from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import (LinearProblem, apply_lifting, assemble_matrix, assemble_vector,
                               create_matrix, create_vector, set_bc)

q = lambda u: 1 + u ** 2

def setup(N, k):
    """Mesh, space, exact solution, source and boundary data of -div(q(u) grad u) = f, u = g on the boundary."""
    msh = mesh.create_unit_square(MPI.COMM_WORLD, N, N)
    V = fem.functionspace(msh, ("Lagrange", k))
    x = ufl.SpatialCoordinate(msh)
    u_ex = 1 + ufl.sin(ufl.pi * x[0]) * x[1] ** 2
    f = -ufl.div(q(u_ex) * ufl.grad(u_ex))
    msh.topology.create_connectivity(1, 2)
    dofs = fem.locate_dofs_topological(V, 1, mesh.exterior_facet_indices(msh.topology))
    g = fem.Function(V); g.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    return msh, V, u_ex, f, dofs, g

def l2_error(msh, uh, u_ex):
    return np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form((uh - u_ex) ** 2 * ufl.dx)), op=MPI.SUM))

msh, V, u_ex, f, dofs, g = setup(8, 1)
uh = fem.Function(V); uh.interpolate(lambda x: 1 + x[0] * x[1])          # some state u
v, w = ufl.TestFunction(V), ufl.TrialFunction(V)
F = q(uh) * ufl.inner(ufl.grad(uh), ufl.grad(v)) * ufl.dx - f * v * ufl.dx
J_ufl = ufl.derivative(F, uh, w)
J_hand = q(uh) * ufl.inner(ufl.grad(w), ufl.grad(v)) * ufl.dx + 2 * uh * w * ufl.inner(ufl.grad(uh), ufl.grad(v)) * ufl.dx
dense = lambda form: fem.assemble_matrix(fem.form(form)).to_dense()
print("max |J_hand - J_ufl| =", np.abs(dense(J_hand) - dense(J_ufl)).max())

# Taylor test in the direction dw
dw = fem.Function(V); dw.interpolate(lambda x: np.sin(3 * x[0]) * np.cos(2 * x[1]))
Jm = dense(J_ufl)
u0 = uh.x.array.copy()
vec = lambda: fem.assemble_vector(fem.form(F)).array.copy()
F0 = vec()
print(" eps        |F(u+eps w) - F(u) - eps J w|    ratio    without J: |F(u+eps w) - F(u)|")
prev = None
for eps in (1e-1, 5e-2, 2.5e-2, 1.25e-2):
    uh.x.array[:] = u0 + eps * dw.x.array
    Fe = vec()
    r2, r1 = np.linalg.norm(Fe - F0 - eps * Jm @ dw.x.array), np.linalg.norm(Fe - F0)
    print(f"{eps:8.4f}   {r2:.3e}" + (f"                      {prev / r2:5.2f}" if prev else " " * 27) + f"    {r1:.3e}")
    prev = r2
uh.x.array[:] = u0

> **✏️ Your turn (5 min).** Repeat the Taylor test with the **Picard** "Jacobian" (without the $2u\,w\,\nabla u\cdot\nabla v$ term). Which order do you get,
> and why does that mean Picard cannot converge quadratically?

In [ ]:
%%fenicsx -np 1
# TODO: Taylor test with the Picard matrix

## 2 · Newton by hand  *(lecture §A3, §B1–B2)*
The loop: assemble $J(u^k)$ and $F(u^k)$, lift and `ghostUpdate`, apply the **homogeneous** condition to the correction, solve
$J\,\delta u=F$, update $u^{k+1}=u^k-\delta u$. Initial guess: the solution with $q=1$. Two processes.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import (LinearProblem, apply_lifting, assemble_matrix, assemble_vector,
                               create_matrix, create_vector, set_bc)

q = lambda u: 1 + u ** 2

def setup(N, k):
    """Mesh, space, exact solution, source and boundary data of -div(q(u) grad u) = f, u = g on the boundary."""
    msh = mesh.create_unit_square(MPI.COMM_WORLD, N, N)
    V = fem.functionspace(msh, ("Lagrange", k))
    x = ufl.SpatialCoordinate(msh)
    u_ex = 1 + ufl.sin(ufl.pi * x[0]) * x[1] ** 2
    f = -ufl.div(q(u_ex) * ufl.grad(u_ex))
    msh.topology.create_connectivity(1, 2)
    dofs = fem.locate_dofs_topological(V, 1, mesh.exterior_facet_indices(msh.topology))
    g = fem.Function(V); g.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    return msh, V, u_ex, f, dofs, g

def l2_error(msh, uh, u_ex):
    return np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form((uh - u_ex) ** 2 * ufl.dx)), op=MPI.SUM))

def solve_nonlinear(N, k, jacobian="newton", guess="linear", tol=1e-12, max_it=50):
    """Newton (jacobian='newton') or Picard (jacobian='picard'). Returns (u_h, history [(k, |F|, |du|)], msh, u_ex)."""
    msh, V, u_ex, f, dofs, g = setup(N, k)
    uh, du = fem.Function(V), fem.Function(V)
    v, w = ufl.TestFunction(V), ufl.TrialFunction(V)
    if guess == "linear":                       # u^0 = solution with q = 1 (satisfies u = g on the boundary)
        uh.x.array[:] = LinearProblem(ufl.inner(ufl.grad(w), ufl.grad(v)) * ufl.dx, f * v * ufl.dx,
                                      bcs=[fem.dirichletbc(g, dofs)], petsc_options_prefix=f"guess_{N}_{k}_{jacobian}_",
                                      petsc_options={"ksp_type": "preonly", "pc_type": "lu"}).solve().x.array
    else:                                       # u^0 = g on the boundary, 0 inside
        uh.x.array[:] = 0.0
        fem.set_bc(uh.x.array, [fem.dirichletbc(g, dofs)])
    F = q(uh) * ufl.inner(ufl.grad(uh), ufl.grad(v)) * ufl.dx - f * v * ufl.dx
    J = ufl.derivative(F, uh, w) if jacobian == "newton" else q(uh) * ufl.inner(ufl.grad(w), ufl.grad(v)) * ufl.dx
    Fc, Jc = fem.form(F), fem.form(J)
    bc0 = fem.dirichletbc(0.0, dofs, V)         # the correction vanishes on the boundary
    A, b = create_matrix(Jc), create_vector(V)
    ksp = PETSc.KSP().create(msh.comm)
    ksp.setOperators(A); ksp.setType("preonly"); ksp.getPC().setType("lu")
    history = []
    for it in range(1, max_it + 1):
        A.zeroEntries(); assemble_matrix(A, Jc, bcs=[bc0]); A.assemble()
        with b.localForm() as bl:
            bl.set(0.0)
        assemble_vector(b, Fc)
        apply_lifting(b, [Jc], bcs=[[bc0]])
        b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
        set_bc(b, [bc0])
        rnorm = b.norm()                        # collective
        ksp.solve(b, du.x.petsc_vec); du.x.scatter_forward()
        uh.x.array[:] -= du.x.array             # J du = F,  u <- u - du
        dnorm = du.x.petsc_vec.norm()           # collective
        history.append((it, rnorm, dnorm))
        if dnorm < tol:
            break
    else:
        raise RuntimeError(f"no convergence in {max_it} iterations")
    ksp.destroy(); A.destroy(); b.destroy()
    return uh, history, msh, u_ex

uh, hist, msh, u_ex = solve_nonlinear(32, 1, "newton")
d = np.array([h[2] for h in hist])
if msh.comm.rank == 0:
    for it, r, dn in hist:
        print(f"k={it}  |F(u^k)| = {r:.3e}   |du| = {dn:.3e}")
    print("|du_k+1| / |du_k|^2 =", np.round(d[1:] / d[:-1] ** 2, 3))
err = l2_error(msh, uh, u_ex)
if msh.comm.rank == 0:
    print(f"L2 error of the converged solution: {err:.3e}")
    np.save("/tmp/fnx/newton_hist.npy", np.array(hist))

> **✏️ Your turn (5 min).** Run the same with `jacobian="picard"`. Count the iterations, compute $\|\delta u_{k+1}\|/\|\delta u_k\|$, and compare the final
> $L^2$ error with Newton's. Save the history to `/tmp/fnx/picard_hist.npy` and plot both histories (semilog) in the next cell.

In [ ]:
%%fenicsx -np 2
# TODO: Picard iteration, history to /tmp/fnx/picard_hist.npy

In [ ]:
# TODO: semilog plot of |du| against k for Newton and Picard

## 3 · Initial guess, mesh independence and rates  *(lecture §A5–A6)*
Newton from the linear initial guess on $N=8,\dots,64$, P1 and P2: the number of iterations, the $L^2$ error of the converged solution and its rate.

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import (LinearProblem, apply_lifting, assemble_matrix, assemble_vector,
                               create_matrix, create_vector, set_bc)

q = lambda u: 1 + u ** 2

def setup(N, k):
    """Mesh, space, exact solution, source and boundary data of -div(q(u) grad u) = f, u = g on the boundary."""
    msh = mesh.create_unit_square(MPI.COMM_WORLD, N, N)
    V = fem.functionspace(msh, ("Lagrange", k))
    x = ufl.SpatialCoordinate(msh)
    u_ex = 1 + ufl.sin(ufl.pi * x[0]) * x[1] ** 2
    f = -ufl.div(q(u_ex) * ufl.grad(u_ex))
    msh.topology.create_connectivity(1, 2)
    dofs = fem.locate_dofs_topological(V, 1, mesh.exterior_facet_indices(msh.topology))
    g = fem.Function(V); g.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    return msh, V, u_ex, f, dofs, g

def l2_error(msh, uh, u_ex):
    return np.sqrt(msh.comm.allreduce(fem.assemble_scalar(fem.form((uh - u_ex) ** 2 * ufl.dx)), op=MPI.SUM))

def solve_nonlinear(N, k, jacobian="newton", guess="linear", tol=1e-12, max_it=50):
    """Newton (jacobian='newton') or Picard (jacobian='picard'). Returns (u_h, history [(k, |F|, |du|)], msh, u_ex)."""
    msh, V, u_ex, f, dofs, g = setup(N, k)
    uh, du = fem.Function(V), fem.Function(V)
    v, w = ufl.TestFunction(V), ufl.TrialFunction(V)
    if guess == "linear":                       # u^0 = solution with q = 1 (satisfies u = g on the boundary)
        uh.x.array[:] = LinearProblem(ufl.inner(ufl.grad(w), ufl.grad(v)) * ufl.dx, f * v * ufl.dx,
                                      bcs=[fem.dirichletbc(g, dofs)], petsc_options_prefix=f"guess_{N}_{k}_{jacobian}_",
                                      petsc_options={"ksp_type": "preonly", "pc_type": "lu"}).solve().x.array
    else:                                       # u^0 = g on the boundary, 0 inside
        uh.x.array[:] = 0.0
        fem.set_bc(uh.x.array, [fem.dirichletbc(g, dofs)])
    F = q(uh) * ufl.inner(ufl.grad(uh), ufl.grad(v)) * ufl.dx - f * v * ufl.dx
    J = ufl.derivative(F, uh, w) if jacobian == "newton" else q(uh) * ufl.inner(ufl.grad(w), ufl.grad(v)) * ufl.dx
    Fc, Jc = fem.form(F), fem.form(J)
    bc0 = fem.dirichletbc(0.0, dofs, V)         # the correction vanishes on the boundary
    A, b = create_matrix(Jc), create_vector(V)
    ksp = PETSc.KSP().create(msh.comm)
    ksp.setOperators(A); ksp.setType("preonly"); ksp.getPC().setType("lu")
    history = []
    for it in range(1, max_it + 1):
        A.zeroEntries(); assemble_matrix(A, Jc, bcs=[bc0]); A.assemble()
        with b.localForm() as bl:
            bl.set(0.0)
        assemble_vector(b, Fc)
        apply_lifting(b, [Jc], bcs=[[bc0]])
        b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
        set_bc(b, [bc0])
        rnorm = b.norm()                        # collective
        ksp.solve(b, du.x.petsc_vec); du.x.scatter_forward()
        uh.x.array[:] -= du.x.array             # J du = F,  u <- u - du
        dnorm = du.x.petsc_vec.norm()           # collective
        history.append((it, rnorm, dnorm))
        if dnorm < tol:
            break
    else:
        raise RuntimeError(f"no convergence in {max_it} iterations")
    ksp.destroy(); A.destroy(); b.destroy()
    return uh, history, msh, u_ex

for k in (1, 2):
    its, E = [], []
    for N in (8, 16, 32, 64):
        uh, hist, msh, u_ex = solve_nonlinear(N, k)
        its.append(len(hist)); E.append(l2_error(msh, uh, u_ex))
    if MPI.COMM_WORLD.rank == 0:
        print(f"P{k}: Newton iterations {its}  L2 errors {[f'{e:.2e}' for e in E]}  "
              f"rates {np.round(np.log2(np.array(E[:-1]) / np.array(E[1:])), 2)}")

> **✏️ Your turn (5 min).** Start Newton from `guess="zero"` (boundary values, zero inside) for $N=32$, P1. How many iterations now, and in which
> iterations does the quadratic phase start? What would you do for a harder problem where this guess does not converge at all?

In [ ]:
%%fenicsx -np 2
# TODO: Newton from the zero initial guess

## 4 · LLM-assisted coding: live demo  *(lecture Part D)*
Prompt (paste the API reference sheet first):

```text
Context: [API reference sheet]
Task: DOLFINx 0.11, %%fenicsx -np 2 cell. Solve -div((1 + u^2) grad u) = f on the unit square with u = g on the boundary by Newton's method
written by hand (no NonlinearProblem): residual F with a fem.Function, Jacobian with ufl.derivative, PETSc assembly with lifting and
ghostUpdate, LU. Manufactured solution u = 1 + sin(pi x) y^2. Print |F| and |du| per iteration and the L2 error for N = 8, 16, 32 (P1) on rank 0.
```
Paste the result below **unchanged**, run it with `-np 2`, then go through the checklist.

In [ ]:
%%fenicsx -np 2
# TODO: paste the %%fenicsx part your LLM produced (unchanged) and run it

**Verification checklist** — checks with a known answer

- [ ] The residual uses a `fem.Function` and the Jacobian comes from `ufl.derivative` (or passes a Taylor test)
- [ ] The initial guess satisfies $u=g$; the correction gets the **homogeneous** condition
- [ ] $\|\delta u\|$ decreases quadratically in the last steps; the loop raises an error if it does not converge
- [ ] The same histories with `-np 1` and `-np 2` (no vector arithmetic before `ghostUpdate`)
- [ ] The $L^2$ errors converge at rate 2 for P1